# Tutorial: Managing Data in the FNQS Project

This notebook explains how we manage the data in Neural Network Quantum States project, particularly for a Foundational model (FNQS).
It covers how to save configurations, network weights, and MCMC samples, and how to reload them for analysis.
We will simulate the training of a 
To understand the details of the ... please refer to [Other Notebook].

## Part 1: Saving the Data (During Training)
The way we save data consists into two separate form : a "meta" json document, that contains the configuration of our Neural Network model and that is easily readble for a human ; and a .nk file containing the result of the training : the weights in the NN, this file is not meant to be read by a human but it contains the real information.

### Step 1.1: Saving the Configuration (`meta.json`)
We store the physics parameters and ViT architecture in a dictionary and put it to a JSON file.

In [ ]:
import os
import json
import numpy as np

run_dir = "./Run_1D_L5_FNQS_Example" #Name and path of the file containing the data
os.makedirs(run_dir, exist_ok=True)

meta = {
    "L": 5,
    "nb_spins": 25,
    "graph": "Chain 1D",
    "n_dim": 1,
    "hamiltonian": {
        "type": "Ising Disorder",
        "J": 1.0,
        "h0_train_list": [0.2, 0.6, 1.0, 1.5, 2.0],
        "sigma": 0.1
    },
    "model": "ViTFNQS",
    "vit_config": {
        "num_layers": 2,
        "d_model": 16,
        "heads": 4,
        "b": 1,
        "L_eff": 25
    },
    "nb_training_steps": 300
}

In this example, the meta file contains mostly the structure of the model and the hyperparameters. However, the strenght of that method is to personalise it according to the needs of the project so that the data is always accompagned by any information that could be useful : to reproduce the training, to plot any graphs or to compare the results to another training. 

The following cell is a classic configuration of a ViT NN. The details of this program can be found in this notebook [link].
Notice how we use the meta we defined as a dictionnary to acces data such as the number of spin L or the ViT hyperparameters. 

In [ ]:
import netket as nk
import netket_foundation as nkf
import optax

hi = nk.hilbert.Spin(0.5, meta["L"])
ps = nkf.ParameterSpace(N=hi.size, min=0, max=10)

ma = nkf.model.ViTFNQS(**meta["vit_config"], n_coups=ps.size, complex=True, disorder=True)
sa = nk.sampler.MetropolisLocal(hi, n_chains=16)

total_configs = len(meta["hamiltonian"]["h0_train_list"]) * 2 
vs = nkf.FoundationalQuantumState(sa, ma, ps, n_replicas=total_configs, n_samples=1024)

rng = np.random.default_rng(1)
params_list = np.abs(rng.normal(loc=1.0, scale=meta["hamiltonian"]["sigma"], size=(total_configs, hi.size)))
vs.parameter_array = params_list

def create_operator(params):
    ha_X = sum(params[i] * nkf.operator.sigmax(hi, i) for i in range(hi.size))
    ha_ZZ = sum(nkf.operator.sigmaz(hi, i) @ nkf.operator.sigmaz(hi, (i + 1) % hi.size) for i in range(hi.size))
    return -ha_X - meta["hamiltonian"]["J"] * ha_ZZ

ha_p = nkf.operator.ParametrizedOperator(hi, ps, create_operator)

optimizer = optax.sgd(0.01)
gs = nkf.VMC_NG(ha_p, optimizer, variational_state=vs)

We will now use netket integrated logger. It consists into two type of files : 
- the json logger file that [A COMPLETER]; 
- the .nk files, that contains the optimized weight of our NN. Here, we decided to save every 10 steps, in case the training stops without reaching the whole number of steps, we can still work with what was done.  

In [ ]:
log = nk.logging.JsonLog(os.path.join(run_dir, "log_data"), save_params=False)
callback = nk.callbacks.SaveState(run_dir,prefixe="state",save_every=10)

Now that we defined where we wanted our logger to be. We can launch the training. The next cell runs the optimization process with the number of steps we defined in the meta, with the hamiltonian we constructed in the firsts cells, and its output will be the loggers and the .nk files.

In [ ]:
gs.run(n_iter=meta["nb_training_steps"], out=log, obs={"ham":ha_p}, callback=callback)

## Part 2. Using the data to compute an observable

In [ ]:
import matplotlib.pyplot as plt

log_path = os.path.join(run_dir, "log_data.json")
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        log_data = json.load(f)
    
    # Example: Plotting energy for the first replica
    if "ham" in log_data and len(log_data["ham"]) > 0:
        first_replica_data = log_data["ham"][0]
        iters = first_replica_data["iters"]
        energy_mean = first_replica_data["Mean"]["real"]
        
        plt.figure(figsize=(6, 4))
        plt.plot(iters, energy_mean, label="VMC Energy")
        plt.xlabel("Iterations")
        plt.ylabel("Energy")
        plt.title("Optimization History")
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.show()

In [ ]:
run_dir = "logs/Run_1D_L5_FNQS_Example" #Name and path of the file containing the data

meta_path = os.path.join(run_dir, "meta.json")
with open(meta_path, 'r') as f:
    meta = json.load(f)

L = meta["L"]
sigma_disorder = meta["hamiltonian"]["sigma"]
h0_train_list = meta["hamiltonian"]["h0_train_list"]
vit_params = meta["vit_config"]

Now that we have the metadata of the training, we can load the trained network and run a test through it to plot interesting observables.

In [ ]:
log = nk.logging.JsonLog(f"{run_dir}/log_data.json")

Here is a simple example to show how to plot the energy by number , wich is a data that was already

In [ ]:
plt.figure(figsize=(10, 5))

for i in range(meta["total_configs_train"]):
    energy = log.data["ham"][i].Mean.real
    iters = log.data["ham"][i].iters
    
    plt.plot(iters, energy, alpha=0.3, label=f"Rep {i}" if i < 5 else "")

plt.xlabel("Steps")
plt.ylabel("Energy $\langle H \rangle$")
plt.title("Energy convergence for all configurations")
plt.grid(True)
plt.show()

In [ ]:
import glob

checkpoints = glob.glob(os.path.join(run_dir, "*.nk"))
assert checkpoints, "No file checkpoint found."
checkpoint_path = sorted(checkpoints, key=lambda x: int(x.split('_')[-1].split('.')[0]))[-1]

We used here the glob library, we search for every .nk file, then we sort it and we take the state corresponding to the highest otimization step.
However, we could also have saved the number of steps [plus précis, traduire itérations peut être] in the meta and loaded it here. 

In [ ]:
hi = nk.hilbert.Spin(0.5, L)
ps = nkf.ParameterSpace(N=L, min=0, max=10*max(h0_train_list + h0_train_list)) #\\! Définir une train liste

ma_test = nkf.model.ViTFNQS(
    **meta["vit_config"],
    n_coups=ps.size,
    complex=True,
    disorder=True
)

sa_test = nk.sampler.MetropolisLocal(hi)
vs_test = nkf.FoundationalQuantumState(sa_test, ma_test, ps, n_replicas=meta["total_configs_train"])
vs_test.load(checkpoint_path)

replica_index = 0
pars_test = np.load(f"{run_dir}/disorder_configs.npy")[replica_index]

We will now plot the longitudinal magnetization:
$$M_z = \frac{1}{L} \sum_{i=1}^{L} \sigma_i^z$$, value of interest for the Ising hamiltonien. 
For each h0 value, we associate the mean of the replicas we drew (it should be near h0 since we drew them using a normal ...) ; and the mean of the magnetization values associated to each replica. 

In [ ]:
mz_vals = []
h_vals = []

mz_op = sum(nkf.operator.sigmaz(hi, i) for i in range(meta["L"])) / meta["L"]

for i in range(total_configs):
    pars = vs_test.parameter_array[i]
    _vs_test = vs_test.get_state(pars)
    
    res = _vs_test.expect(mz_op)
    mz_vals.append(res.Mean.real)
    
    # We use the mean of the parameters as a proxy for the effective transverse field
    h_vals.append(np.mean(pars))

plt.figure(figsize=(8, 5))
plt.scatter(h_vals, np.abs(mz_vals), color='red')
plt.xlabel("Average Transverse field $\\langle h \\rangle$")
plt.ylabel("Magnetization $|M_z|$")
plt.title("Phase transition: Magnetization vs Transverse field ")
plt.grid(True, alpha=0.3)
plt.show()